# 07. ViT-B/16 with Focal Loss and Staged Fine-Tuning

**Objective**  
The objective of this experiment is to improve the weighted ViT baseline by targeting its main weakness: **low melanoma precision despite strong recall**.  
This notebook keeps the same ViT-B/16 backbone and augmentation pipeline, but changes two parts of the training setup:

- **Binary focal loss** to reduce the impact of easy examples and focus learning on harder, more informative cases  
- **Staged fine-tuning** to stabilize optimization by warming up the classification head first, then unfreezing the full backbone with a smaller learning rate  

**Model Architecture**

| Stage | Layers |
| --- | --- |
| Input | RGB dermoscopic image |
| Patch embedding | `16 × 16` image patches |
| Backbone | Pretrained `ViT-B/16` |
| Classification head | Final linear layer with `1` output |
| Output | Single logit for melanoma probability |

### Hypothesis

Compared with the weighted-loss baseline, focal loss should help the model spend less capacity on already easy non-melanoma cases, while staged fine-tuning should make optimization more stable and reduce noisy full-model updates early in training.

### Summary of This Notebook

- Uses the same **pretrained ViT-B/16 model** for binary classification  
- Keeps the existing **augmented training pipeline** from the shared data transforms  
- Replaces weighted BCE with **BinaryFocalLoss(alpha=0.75, gamma=2.0)**  
- Uses a **2-phase training schedule**: head warm-up first, then full fine-tuning with discriminative learning rates


### 1. Setup and Imports

This section loads the project modules, trainer utilities, and shared evaluation helpers, then selects the available compute device (`CUDA`, `MPS`, or `CPU`).  
It also imports the shared focal-loss implementation so the experiment stays aligned with the reusable training code in `src.training.losses`.


In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / 'src').exists())
sys.path.insert(0, str(ROOT))

import torch
import torch.optim as optim

from src.data.dataloader import get_dataloaders
from src.data.transform import get_augmented_train_transforms
from src.models.vit import ViTBinaryClassifier
from src.training.losses import BinaryFocalLoss
from src.training.trainer import train_one_epoch, validate_one_epoch
from src.utils import plot_training_curves, find_best_threshold, evaluate_model

import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


Using device: mps


### 2. Data Split and Focal-Loss Configuration

The dataset is loaded using the predefined train, validation, and test CSV splits.  
This notebook keeps the same augmentation pipeline already used in `01.vit_b16_weighted`, but changes the optimization strategy:

- `ViTBinaryClassifier(pretrained=True, freeze_backbone=True)` for a stable head warm-up phase
- `BinaryFocalLoss(alpha=0.75, gamma=2.0)` to emphasize harder training examples under class imbalance
- `Adam` optimizer with **two phases**:
  - warm-up: train only the head
  - fine-tuning: unfreeze the full model with lower LR on the backbone and higher LR on the head
- Batch size `32` and image size `224 × 224` to match the ViT input format


In [2]:
train_loader, val_loader, test_loader = get_dataloaders(
    train_csv=str(ROOT / "data_new/splits/train.csv"),
    val_csv=str(ROOT / "data_new/splits/val.csv"),
    test_csv=str(ROOT / "data_new/splits/test.csv"),
    image_dir=str(ROOT / "data_new/images/train"),
    test_image_dir=str(ROOT / "data_new/images/test"),
    batch_size=32,
    image_size=224,
    num_workers=0,
    transform_train=get_augmented_train_transforms(image_size=224),
)

warmup_epochs = 2
finetune_epochs = 8
num_epochs = warmup_epochs + finetune_epochs

focal_alpha = 0.75
focal_gamma = 2.0
warmup_lr = 1e-3
backbone_lr = 1e-5
head_lr = 5e-5

model = ViTBinaryClassifier(pretrained=True, freeze_backbone=True).to(device)
criterion = BinaryFocalLoss(alpha=focal_alpha, gamma=focal_gamma)

optimizer = optim.Adam(model.model.heads.head.parameters(), lr=warmup_lr)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Warm-up epochs: {warmup_epochs}")
print(f"Fine-tune epochs: {finetune_epochs}")
print(f"Focal loss alpha: {focal_alpha}")
print(f"Focal loss gamma: {focal_gamma}")
print(f"Trainable params at start: {trainable_params:,} / {total_params:,}")
print("Augmentation check: using get_augmented_train_transforms(image_size=224)")


Warm-up epochs: 2
Fine-tune epochs: 8
Focal loss alpha: 0.75
Focal loss gamma: 2.0
Trainable params at start: 769 / 85,799,425
Augmentation check: using get_augmented_train_transforms(image_size=224)


### 3. Train the ViT-B/16 Model with Staged Fine-Tuning

Training is split into two phases:

- **Warm-up phase**: only the classification head is trainable, so the new head can adapt to the dataset without disturbing pretrained backbone features too early
- **Fine-tuning phase**: the full ViT backbone is unfrozen, but the optimizer uses a smaller learning rate for backbone parameters and a larger one for the head

The checkpoint with the highest validation AUC is saved for later threshold tuning and test evaluation.


In [3]:
best_val_auc = 0.0
train_history, val_history = [], []
unfrozen = False

for epoch in range(num_epochs):
    if epoch == warmup_epochs and not unfrozen:
        for param in model.model.parameters():
            param.requires_grad = True

        backbone_params = [
            param
            for name, param in model.model.named_parameters()
            if not name.startswith('heads.head')
        ]
        head_params = list(model.model.heads.head.parameters())

        optimizer = optim.Adam(
            [
                {"params": backbone_params, "lr": backbone_lr},
                {"params": head_params, "lr": head_lr},
            ]
        )
        unfrozen = True
        print("Unfroze full backbone and switched to discriminative learning rates.")

    phase = "Warm-up" if epoch < warmup_epochs else "Fine-tune"

    train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = validate_one_epoch(model, val_loader, criterion, device)

    train_history.append(train_metrics)
    val_history.append(val_metrics)

    print(f"Epoch [{epoch+1}/{num_epochs}] | Phase: {phase}")
    print(f"  Train | Loss: {train_metrics['loss']:.4f}, Bal Acc: {train_metrics['balanced_accuracy']:.4f}, "
          f"Recall: {train_metrics['recall']:.4f}, F2: {train_metrics['f2']:.4f}, AUC: {train_metrics['auc']:.4f}")
    print(f"  Val   | Loss: {val_metrics['loss']:.4f}, Bal Acc: {val_metrics['balanced_accuracy']:.4f}, "
          f"Recall: {val_metrics['recall']:.4f}, F2: {val_metrics['f2']:.4f}, AUC: {val_metrics['auc']:.4f}")

    if val_metrics['auc'] > best_val_auc:
        best_val_auc = val_metrics['auc']
        torch.save(model.state_dict(), ROOT / 'models/vit_b16_focal_staged_finetune.pth')
        print("  Saved best model.")


KeyboardInterrupt: 

### 4. Plot Training Curves

This notebook uses the shared evaluation utility to render the standard 2 × 2 dashboard: Loss, Balanced Accuracy, Recall, and F2.  
That keeps the staged fine-tuning experiment directly comparable with the weighted ViT baseline.


In [ ]:
plot_training_curves(train_history, val_history)


### 5. Tune the Classification Threshold on the Validation Set

Focal loss changes the training objective, but the final operating point still needs to be selected separately.  
This step reloads the best saved checkpoint and finds the probability threshold that maximizes **F2 score** on the validation set.


In [ ]:
model.load_state_dict(torch.load(ROOT / 'models/vit_b16_focal_staged_finetune.pth', map_location=device))
best_threshold, best_f2 = find_best_threshold(model, val_loader, device)

print(f"Best validation threshold: {best_threshold:.2f}")
print(f"Best validation F2: {best_f2:.4f}")


### 6. Evaluate on the Test Set

The best focal-loss ViT-B/16 checkpoint is evaluated on the held-out test set using the threshold selected on the validation set.  
The shared evaluation utility reports AUC-ROC, balanced accuracy, F2 score, a classification report, and visual diagnostics such as the confusion matrix and ROC curve.


In [ ]:
evaluate_model(model, test_loader, device, threshold=best_threshold)


### Results Summary (ViT-B/16 with Focal Loss + Staged Fine-Tuning)

### 7. Baseline Interpretation

### Key Metrics
- **AUC-ROC:** fill after running  
- **Best Threshold:** fill after running  
- **F2 Score:** fill after running  
- **Balanced Accuracy:** fill after running  

### Class-wise Performance
- **Melanoma Recall:** fill after running  
- **Melanoma Precision:** fill after running  
- **Non-melanoma Precision:** fill after running  

### Key Observations
- Compare melanoma recall against the weighted ViT baseline
- Check whether focal loss reduced false positives and improved melanoma precision
- Check whether staged fine-tuning improved validation stability and test generalization
- Note the trade-off between sensitivity and precision after threshold tuning

### Conclusion
- State whether this is a stronger screening model than `01.vit_b16_weighted`
- Summarize whether focal loss improved the precision-recall balance
- Summarize whether staged fine-tuning was useful for optimization stability
- Future improvements should focus on:
  - Further improving precision without losing too much recall
  - Better calibration or threshold selection if needed
